# Glass Type Classification

UCI Glass Identification dataset: 214 samples, 9 chemical features, 6 classes
(types 1, 2, 3, 5, 6, 7 — type 4 does not occur in the data).

- 1: building windows, float processed
- 2: building windows, non-float processed
- 3: vehicle windows, float processed
- 5: containers
- 6: tableware
- 7: headlamps

Pipeline: load data -> compare models with repeated CV on a training split ->
tune the best families -> evaluate the selected model once on a held-out test set.

In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import (GridSearchCV, RepeatedStratifiedKFold,
                                     StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")
SEED = 42

## Load data

CSV has no header: RI, Na, Mg, Al, Si, K, Ca, Ba, Fe, Type.

In [2]:
COLS = ["RI", "Na", "Mg", "Al", "Si", "K", "Ca", "Ba", "Fe", "Type"]
NAMES = {1: "building_float", 2: "building_non_float", 3: "vehicle_float",
         5: "containers", 6: "tableware", 7: "headlamps"}

path = "glass.csv"  # change this if your file is named/located differently
df = pd.read_csv(path, header=None, names=COLS)
X, y = df.drop(columns="Type"), df["Type"]

print(f"Rows: {len(df)}, missing values: {int(df.isna().sum().sum())}")
df.head()

Rows: 214, missing values: 0


,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


In [3]:
y.value_counts().sort_index().rename(NAMES).to_frame("count")

,count
Type,
building_float,70
building_non_float,76
vehicle_float,17
containers,13
tableware,9
headlamps,29


## Train / test split

Stratified split; the test set stays untouched until the final evaluation.

In [4]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)
X_tr.shape, X_te.shape

((160, 9), (54, 9))

## Compare candidate models

5x5 repeated stratified CV, computed only on the training split.

In [5]:
candidates = {
    "kNN": Pipeline([("s", StandardScaler()),
                     ("m", KNeighborsClassifier(n_neighbors=3, weights="distance"))]),
    "SVM (RBF)": Pipeline([("s", StandardScaler()),
                           ("m", SVC(C=10, gamma="scale", class_weight="balanced"))]),
    "Random Forest": RandomForestClassifier(
        n_estimators=500, class_weight="balanced", random_state=SEED),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=500, class_weight="balanced", random_state=SEED),
}

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)
scores = {}
for name, model in candidates.items():
    s = cross_val_score(model, X_tr, y_tr, cv=cv, scoring="accuracy")
    scores[name] = s.mean()
    print(f"{name:15s} {s.mean():.3f} +/- {s.std():.3f}")

kNN             0.670 +/- 0.080


SVM (RBF)       0.654 +/- 0.055


Random Forest   0.762 +/- 0.061


Extra Trees     0.766 +/- 0.068


## Tune the two strongest families

In [6]:
grids = {
    "Random Forest": (RandomForestClassifier(class_weight="balanced", random_state=SEED),
                      {"n_estimators": [300, 600], "max_depth": [None, 8, 12],
                       "min_samples_leaf": [1, 2], "max_features": ["sqrt", 0.5]}),
    "Extra Trees": (ExtraTreesClassifier(class_weight="balanced", random_state=SEED),
                    {"n_estimators": [300, 600], "max_depth": [None, 8, 12],
                     "min_samples_leaf": [1, 2], "max_features": ["sqrt", 0.5]}),
}
inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
tuned = {}
for name, (est, grid) in grids.items():
    gs = GridSearchCV(est, grid, cv=inner, scoring="accuracy", n_jobs=-1).fit(X_tr, y_tr)
    tuned[name] = gs
    print(f"{name:15s} best CV acc {gs.best_score_:.3f}  params {gs.best_params_}")

best_name = max(tuned, key=lambda k: tuned[k].best_score_)
best = tuned[best_name].best_estimator_
print(f"\nSelected model: {best_name}")

Random Forest   best CV acc 0.762  params {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'n_estimators': 300}


Extra Trees     best CV acc 0.775  params {'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 300}

Selected model: Extra Trees


## Final evaluation on the held-out test set

This is the single, final look at the test set.

In [7]:
pred = best.predict(X_te)
acc = (pred == y_te).mean()
print(f"Test accuracy: {acc:.3f} on {len(y_te)} samples\n")

labels = sorted(NAMES)
print(classification_report(y_te, pred, labels=labels,
                            target_names=[NAMES[l] for l in labels], zero_division=0))

Test accuracy: 0.870 on 54 samples

                    precision    recall  f1-score   support

    building_float       0.80      0.89      0.84        18
building_non_float       0.94      0.84      0.89        19
     vehicle_float       0.67      0.50      0.57         4
        containers       1.00      1.00      1.00         3
         tableware       0.67      1.00      0.80         2
         headlamps       1.00      1.00      1.00         8

          accuracy                           0.87        54
         macro avg       0.85      0.87      0.85        54
      weighted avg       0.88      0.87      0.87        54



In [8]:
cm = pd.DataFrame(confusion_matrix(y_te, pred, labels=labels),
                  index=[f"true_{l}" for l in labels],
                  columns=[f"pred_{l}" for l in labels])
cm

,pred_1,pred_2,pred_3,pred_5,pred_6,pred_7
true_1,16,1,1,0,0,0
true_2,2,16,0,0,1,0
true_3,2,0,2,0,0,0
true_5,0,0,0,3,0,0
true_6,0,0,0,0,2,0
true_7,0,0,0,0,0,8


## Feature importance

In [9]:
imp = pd.Series(best.feature_importances_, index=X.columns).sort_values(ascending=False)
imp.round(3)

Mg    0.191
Ca    0.130
K     0.124
Al    0.120
Na    0.116
RI    0.109
Ba    0.082
Si    0.082
Fe    0.045
dtype: float64

## Notes / caveats

- The test set is small (54 samples), and the rarer classes (3, 5, 6) have only
  a handful of samples in it, so the 0.870 test accuracy is noisy. The 5x5 CV
  accuracy (about 0.77) is the more stable estimate of real-world performance.
- Class 1 (building, float) and class 2 (building, non-float) are the hardest
  to tell apart — this is a known property of this dataset, not a bug.

## Final live model (trained on all data)

The evaluation above uses a held-out test set, which is correct for measuring
real-world accuracy. For the deployed live-prediction dashboard, a separate
model is trained on the *full* 214-sample dataset (no held-out split) so it
uses all available information for actual predictions. Same algorithm and
similar hyperparameters as the tuned model above.

In [10]:
import json

final_model = ExtraTreesClassifier(n_estimators=150, max_depth=10, min_samples_leaf=2,
                                    class_weight="balanced", random_state=42)
final_model.fit(X, y)
classes = final_model.classes_.tolist()
print("Class order:", classes)

Class order: [1, 2, 3, 5, 6, 7]


### Export tree structure to JSON

This is the exact format embedded in the dashboard's HTML/JS, so the browser can replicate `predict_proba` with plain JavaScript, entirely client-side.

In [11]:
trees_json = []
for est in final_model.estimators_:
    t = est.tree_
    n = t.node_count
    left = t.children_left.tolist()
    right = t.children_right.tolist()
    feat = t.feature.tolist()
    thresh = [round(float(v), 6) for v in t.threshold]
    val = t.value.reshape(n, -1)
    val_norm = (val / val.sum(axis=1, keepdims=True)).tolist()
    val_norm = [[round(x, 4) for x in row] for row in val_norm]
    trees_json.append({"L": left, "R": right, "F": feat, "T": thresh, "V": val_norm})

model_export = {
    "classes": classes,
    "feature_names": list(X.columns),
    "trees": trees_json,
}
with open("model_export.json", "w") as f:
    json.dump(model_export, f)

import os
print(f"Exported model_export.json ({os.path.getsize('model_export.json')/1024:.1f} KB, "
      f"{len(trees_json)} trees)")

Exported model_export.json (788.6 KB, 150 trees)


### Sanity check: JSON-based inference matches `predict_proba` exactly

In [12]:
def predict_row_from_json(row, model):
    probs = np.zeros(len(model["classes"]))
    for tr in model["trees"]:
        node = 0
        while tr["L"][node] != -1:
            f = tr["F"][node]
            node = tr["L"][node] if row[f] <= tr["T"][node] else tr["R"][node]
        probs += np.array(tr["V"][node])
    return probs / len(model["trees"])

with open("model_export.json") as f:
    m = json.load(f)

ok = True
for i in [0, 50, 100, 150, 200]:
    row = X.values[i]
    p_sklearn = final_model.predict_proba([row])[0]
    p_manual = predict_row_from_json(row, m)
    match = np.allclose(p_sklearn, p_manual, atol=1e-3)
    ok &= match
    print(i, "match:", match)
print("\nAll checked rows match:" , ok)

0 match: True
50 match: True
100 match: True
150 match: True
200 match: True

All checked rows match: True
